# RandomForestClassifier Hyperparameter Selection

In [2]:
import sys
from pathlib import Path

# Set root
cwd = Path.cwd()
ROOT = next(p for p in [cwd, *cwd.parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.preprocessing import load_model_data
from src.evaluation import splits, leakage
from src.evaluation.make_splits import load_split, SPLITS

SPLIT_LIST = list(SPLITS)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score


In [3]:
# ===== Settings: change these, then Restart & Run All =====
SPLIT = "10km"            # "10km", "10km_subbasin", "7.5km_subbasin", "5km_subbasin"
USE_SUBBASIN = False      # True: subbasin features replace elevation
MODEL_SEED = 0
THRESHOLD_MG_L = 10
DROP_ELEVATION = False      # True: remove elevation_m

In [4]:
# Load the cleaned/merged dataset

X, y, meta = load_model_data(subbasin=USE_SUBBASIN, drop_elevation=DROP_ELEVATION)
FEATURES = list(X.columns)

data = load_split(pd.concat([meta, X], axis=1).assign(log_nitrate=y), split=SPLIT)
print(f"split={SPLIT} | subbasin features={USE_SUBBASIN} | {len(FEATURES)} features")
print(data["fold"].value_counts().sort_index().to_dict(), "| holdout rows:", int(data["holdout"].sum()))



split=10km | subbasin features=False | 50 features
{0: 1035, 1: 972, 2: 1143, 3: 1041, 4: 1089} | holdout rows: 1089


In [5]:
#create subfolds
from src.models import random_forest
development, X_dev, y_dev = random_forest.make_inner_subfolds(data, FEATURES, N_SUBFOLDS = 4, THRESHOLD_MG_L = 10, SEED = 77)

subfold    0    1    2    3
fold                       
0        187  195  237  416
1        109  194  230  439
2        408  258  302  175
3        186  160  224  471


In [5]:
param_grid = {
"n_estimators": [10, 50, 100, 300, 500],
"max_features": [1.0, "sqrt", "log2", 0.5],  
"max_depth" : [i for i in range(1,20)]
}

In [15]:
from sklearn.model_selection import GridSearchCV # type: ignore

def make_search_RFC(inner_cv, param_grid, SEED):
    return GridSearchCV(
        estimator=RandomForestClassifier(
            random_state=SEED,
            n_jobs=1,
            class_weight="balanced"
        ),
        param_grid=param_grid,
        scoring="average_precision",
        cv=inner_cv,
        refit=True,
        n_jobs=-1,
        return_train_score=False,
    )

In [18]:
outer_results, best_params_by_fold, oof_pred = random_forest.fit_classifier_with_nested_cv(
    development, X_dev, y_dev, param_grid, make_search_RFC, N_SUBFOLDS = 4, SEED = 77, 
    THRESHOLD_MG_L = 10, CLASSIFICATION_THRESHOLD=0.5, PRINT=True)

Nested CV results:
 outer_fold  average_precision  roc_auc  precision  recall    f1  balanced_accuracy  brier_score_loss  best_inner_ap  n_exceedances  n_samples  inner_fold_0_ap  inner_fold_1_ap  inner_fold_2_ap  inner_fold_3_ap
          0              0.588    0.812      0.583   0.710 0.640              0.732             0.161          0.629            338       1035            0.601            0.716            0.569            0.630
          1              0.596    0.756      0.482   0.745 0.586              0.667             0.210          0.602            330        972            0.573            0.642            0.557            0.635
          2              0.548    0.745      0.484   0.942 0.640              0.703             0.206          0.571            398       1143            0.471            0.627            0.554            0.634
          3              0.581    0.792      0.570   0.798 0.665              0.745             0.172          0.595            352      